In [62]:
USE [sample];
GO

/*
    Best Practice: Stop "rows affected" noise
*/
SET NOCOUNT ON;

SELECT DB_NAME() AS db_name;
DECLARE @msg VARCHAR(MAX) = 'Initial Database Context: ' + DB_NAME();
EXEC dbo.spInfo @msg
GO


Commands completed successfully.

2026-03-26 07:02:17 | INFO | Initial Database Context: sample

db_name
-------
sample 
(1 row)

Timestamp               | Level | Message                         
------------------------+-------+---------------------------------
2026-03-26 07:02:17.903 | INFO  | Initial Database Context: sample
(1 row)

In [63]:
/*
************************************************************************************
    create genders if tables is empty
************************************************************************************
*/
IF NOT EXISTS (SELECT * FROM dbo.tblGender)         -- check fo emptiness
    BEGIN
        INSERT INTO dbo.tblGender (ID, Gender) 
        VALUES 
            (1, 'Male'), 
            (2, 'Female'), 
            (3, 'Unknown');
        EXEC dbo.spInfo 'INFO | Data inserted into dbo.tblGender.';
    END
ELSE
    BEGIN
        EXEC dbo.spInfo 'Data already inserted into dbo.tblGender.';
    END
GO


2026-03-26 07:02:17 | INFO | INFO | Data inserted into dbo.tblGender.

Timestamp               | Level | Message                                 
------------------------+-------+-----------------------------------------
2026-03-26 07:02:17.937 | INFO  | INFO | Data inserted into dbo.tblGender.
(1 row)

In [64]:
/*
************************************************************************************
    Using the 3-part name (Database.Schema.Table)
    'dbo' is the default schema (Database Owner)
    Schemas act like namespaces (e.g. Sales.Table vs HR.Table)
************************************************************************************
*/
SELECT * FROM sample.dbo.tblGender;
GO


Commands completed successfully.

ID | Gender 
---+--------
1  | Male   
2  | Female 
3  | Unknown
(3 rows)

In [67]:
/*
************************************************************************************
    testing fk
************************************************************************************
*/
DELETE FROM sample.dbo.tblPerson;
SELECT * FROM sample.dbo.tblPerson


Commands completed successfully.

(0 rows)

In [68]:
/*
************************************************************************************
    missing values (nulls) are allowed, no fk violation
************************************************************************************
*/
INSERT INTO sample.dbo.tblPerson
(ID, Name, Email)                       -- need to specify columns due to missing gender value
VALUES (1, 'john', 'j@j.com');          -- no gender
SELECT *  FROM sample.dbo.tblPerson;


Commands completed successfully.

ID | Name | Email   | GenderId
---+------+---------+---------
1  | john | j@j.com | NULL    
(1 row)

In [69]:
/*
************************************************************************************
    illegal gender values are not allowed, fk violation
************************************************************************************
*/
BEGIN TRY
    INSERT INTO sample.dbo.tblPerson
    -- (ID, Name, Email)
    VALUES (2, 'mary', 'm@m.com', 99);      -- illegal gender
END TRY
BEGIN CATCH
    -- SELECT ERROR_NUMBER(), ERROR_MESSAGE();
    EXEC dbo.spError 'An error occurred. Execution jumped to the CATCH block.'
END CATCH


2026-03-26 07:05:00 | ERROR | An error occurred. Execution jumped to the CATCH block.: 547 - The INSERT statement conflicted with the FOREIGN KEY constraint "FK_tblPerson_tblGender". The conflict occurred in database "sample", table "dbo.tblGender", column 'ID'.

Timestamp               | Level | Message                                                                                                                                                                                                 
------------------------+-------+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
2026-03-26 07:05:00.533 | ERROR | An error occurred. Execution jumped to the CATCH block.: 547 - The INSERT statement conflicted with the FOREIGN KEY constraint "FK_tblPerson_tblGender". The conflict occurred in database "sample", table "dbo.tblGender", column 'ID'.
(1 row)

In [70]:
INSERT INTO sample.dbo.tblPerson
VALUES (2, 'mary', 'm@m.com', 2);      -- correct gender
SELECT * FROM  sample.dbo.tblPerson;



Commands completed successfully.

ID | Name | Email   | GenderId
---+------+---------+---------
1  | john | j@j.com | NULL    
2  | mary | m@m.com | 2       
(2 rows)

In [71]:
/*
************************************************************************************
    test each genderid value against tblgender.ID
************************************************************************************
*/
SELECT
    *,
    CASE 
        WHEN GenderID IN (SELECT ID FROM sample.dbo.tblGender) THEN 'Valid'
        ELSE 'Invalid'
    END AS GenderStatus
FROM 
    sample.dbo.tblPerson;
GO

Commands completed successfully.

ID | Name | Email   | GenderId | GenderStatus
---+------+---------+----------+-------------
1  | john | j@j.com | NULL     | Invalid     
2  | mary | m@m.com | 2        | Valid       
(2 rows)